# Module 4.4: Assemble Your Decoder-Only GPT

You have now built **every part** of a modern LLM:

- a **tokenizer** and an **embedding** table (Module 2.1),
- **RoPE** positional rotation (Module 2.2),
- **multi-head causal attention** (Module 3.2),
- **RMSNorm** and the **SwiGLU** feed-forward (Module 4.1),
- the **pre-norm decoder block** wiring (Modules 4.1–4.2).

Everything is a part on the workbench. This notebook is where you **bolt them
together into the actual decoder-only GPT** — the same class you'll train in the
capstone (Module 5.3) and that ships in `src/llm_workout/`. Up to now the assembly
notebooks used *mock* attention to isolate one idea at a time. Here we use the
**real** components, wire the full model ourselves, and then **prove** — with
`torch.allclose` — that what we built is byte-for-byte the library's `GPT`.

No more "import it and trust me." By the end of this page, `from llm_workout.model
import GPT` means *importing your own work*.

## 1. The blueprint

A decoder-only GPT is a short pipeline with a tall stack in the middle:

```mermaid
flowchart TD
    ids["token ids<br/>(B, T)"] --> emb["token embedding<br/>lookup (2.1)"]
    emb --> b1["Decoder block 1"]
    b1 --> b2["Decoder block 2"]
    b2 --> bd["... x N blocks (4.1-4.2)<br/>each: RMSNorm -> attention -> +residual<br/>RMSNorm -> SwiGLU -> +residual"]
    bd --> fn["final RMSNorm"]
    fn --> head["lm_head (Linear)<br/>-> logits (B, T, vocab)"]
    rope["RoPE angles (2.2)"] -.->|"rotate Q,K"| bd
    mask["causal mask (3.1)"] -.->|"block the future"| bd
```

Two things flow *into* the blocks rather than through the front door:

- **RoPE angles** — precomputed once, applied to Q and K inside each attention (2.2).
- **The causal mask** — a lower-triangular block so no position sees the future (3.1).

That's the whole architecture. Let's build it.

## 2. One new idea: weight tying

There's a single concept we haven't met yet, and the model uses it: **weight tying**.

- The **embedding** turns a token id into a vector: it's a `(vocab, d_model)` table,
  one row per token.
- The **`lm_head`** does the opposite at the end — turns a vector back into a score
  per token: a `(d_model, vocab)` matrix.

These two are mirror images: "id → vector" and "vector → id-scores." **Weight tying**
makes them literally *the same matrix* (one is the transpose of the other). Why?

1. It **saves parameters** — for a 50k vocab and `d_model=512` that's ~25M weights
   you don't duplicate.
2. It usually **helps quality** — the notion of "what this token means" (input side)
   and "when to predict this token" (output side) are related, so sharing forces them
   to stay consistent.

In code it's one line: `self.token_embedding.weight = self.lm_head.weight`. GPT-2 and
many models tie; Llama-3 doesn't. Ours does — watch for that line below.

## 3. Import the parts you built

We import the **leaf layers** from the library — these are exactly the components you
implemented in Modules 2–4. Our job here is the **wiring**, not re-deriving the parts.

In [ ]:
import torch
import torch.nn as nn

# The parts you built, in earlier notebooks:
from llm_workout.layers import (
    RMSNorm,                    # Module 4.1
    MultiHeadCausalAttention,   # Module 3.2 (with RoPE + KV cache)
    SwiGLU,                     # Module 4.1
    precompute_freqs_cis,       # Module 2.2 (RoPE angles)
)

torch.manual_seed(0)
print("Imported the leaf components you built in Modules 2-4.")

## 4. The decoder block (the wiring from 4.1–4.2)

A block is two pre-norm sub-layers, each with a residual: `x + attention(norm(x))`,
then `h + ffn(norm(h))`. This is the picture from Module 4.1, now with the **real**
attention instead of a mock.

In [ ]:
class MyDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim):
        super().__init__()
        self.attn      = MultiHeadCausalAttention(d_model, num_heads)
        self.attn_norm = RMSNorm(d_model)
        self.ffn       = SwiGLU(d_model, hidden_dim)
        self.ffn_norm  = RMSNorm(d_model)

    def forward(self, x, mask, freqs_cis):
        # pre-norm attention, then a residual highway
        attn_out, _ = self.attn(self.attn_norm(x), mask=mask, freqs_cis=freqs_cis)
        h = x + attn_out
        # pre-norm FFN, then a second residual highway
        out = h + self.ffn(self.ffn_norm(h))
        return out

print("Decoder block wired: RMSNorm -> attention -> +x, then RMSNorm -> SwiGLU -> +h.")

## 5. The full GPT

Now the outer shell: embedding in front, a stack of `N` blocks, a final norm, and the
`lm_head` — plus the two things that feed the blocks (RoPE angles and the causal mask),
and the **weight-tying** line from section 2.

Read `forward` top to bottom; it's the whole model in a dozen lines.

In [ ]:
class MyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, hidden_dim, max_seq_len=1024):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)          # id -> vector (2.1)

        # RoPE angles, precomputed once for every position, stored as a buffer (2.2)
        freqs_cis = precompute_freqs_cis(d_model // num_heads, max_seq_len)
        self.register_buffer("freqs_cis", freqs_cis)

        self.layers = nn.ModuleList([                                     # the tall stack (4.1-4.2)
            MyDecoderBlock(d_model, num_heads, hidden_dim) for _ in range(num_layers)
        ])
        self.final_norm = RMSNorm(d_model)                               # one last normalize
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)        # vector -> id-scores

        self.token_embedding.weight = self.lm_head.weight                # WEIGHT TYING (section 2)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.token_embedding(idx)                                    # (B, T, d_model)
        freqs_cis = self.freqs_cis[:T]                                   # RoPE angles for these positions

        # causal mask: lower-triangular, so position i can only see 0..i (3.1)
        mask = torch.tril(torch.ones(T, T, device=idx.device)).view(1, 1, T, T)

        for layer in self.layers:                                       # through every block
            x = layer(x, mask, freqs_cis)

        x = self.final_norm(x)
        logits = self.lm_head(x)                                         # (B, T, vocab)

        loss = None
        if targets is not None:
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)      # flatten (Module 5.4 / 5.1)
            )
        return logits, loss

print("Full GPT assembled: embedding -> N blocks -> final norm -> lm_head (weights tied).")

## 6. Proof: you built the real thing

Talk is cheap — let's **prove** `MyGPT` is the library's `GPT`. We build one of each with
the same config, copy the library's trained-in weights into ours, and check that a
forward pass produces **identical** logits and loss.

In [ ]:
from llm_workout.model import GPT

cfg = dict(vocab_size=65, d_model=64, num_layers=3, num_heads=4, hidden_dim=256, max_seq_len=128)

lib = GPT(**cfg)          # the library model
mine = MyGPT(**cfg)       # the one you just assembled

# Same parameter names + structure -> the library's weights drop straight in.
mine.load_state_dict(lib.state_dict())

lib.eval(); mine.eval()
idx     = torch.randint(0, 65, (2, 16))
targets = torch.randint(0, 65, (2, 16))

with torch.no_grad():
    lib_logits,  lib_loss,  _ = lib(idx, targets)   # library returns (logits, loss, kv_caches)
    my_logits,   my_loss      = mine(idx, targets)

print("logits match: ", torch.allclose(lib_logits, my_logits, atol=1e-6))
print("loss match:   ", torch.allclose(lib_loss,  my_loss,  atol=1e-6))
print(f"max logit difference: {(lib_logits - my_logits).abs().max().item():.2e}")
print("\nSame weights in, same numbers out. `llm_workout.model.GPT` IS your MyGPT.")

## 7. It runs — generate a token

An untrained model outputs noise (we haven't trained it — that's the capstone), but it
*works*: feed it ids, get a next-token distribution, sample.

In [ ]:
mine.eval()
prompt = torch.randint(0, 65, (1, 5))            # 5 random "tokens"
with torch.no_grad():
    logits, _ = mine(prompt)
next_probs = torch.softmax(logits[0, -1], dim=-1)  # distribution over the next token (Module 5.5-preview)
next_id = torch.multinomial(next_probs, 1).item()

n_params = sum(p.numel() for p in mine.parameters())
print(f"Your assembled model has {n_params:,} parameters.")
print(f"Given a 5-token prompt, it predicts a next-token id: {next_id}")
print("It's untrained, so that's a random guess -- but the machine is complete and running.")

## Summary

You just **assembled the real decoder-only GPT from the parts you built** — embedding,
RoPE, causal attention, RMSNorm, SwiGLU, residual wiring, a weight-tied `lm_head` — and
proved it is identical to `llm_workout.model.GPT`. Two things to carry forward:

- **The import is no longer a black box.** When the next notebooks write `from
  llm_workout.model import GPT`, that's the class you see above.
- **You learned weight tying** — the embedding and the output head are one shared matrix.

Next, in **Module 5.1**, we give this model a target to learn (the cross-entropy loss),
then in **5.2–5.3** we train *this exact model* on real text.

### 🏋️ Try it yourself

1. **Break the tie.** Delete the `self.token_embedding.weight = self.lm_head.weight` line,
   rebuild `MyGPT`, and print the parameter count. How many *extra* parameters did the tie
   save (it's `vocab_size x d_model`)? Does `load_state_dict` still succeed?
2. **Break the mask.** In `forward`, replace the causal `mask` with `None` (attention then
   sees the whole sequence). Re-run the proof cell — do the logits still match the library?
   Why not? (Which module explained why a decoder *must* mask?)
3. **Grow it.** Build `MyGPT` with `num_layers=6, d_model=128` and print the parameter
   count. Roughly how does it scale with depth and width?

In [ ]:
# Room to experiment.

# Task 1: how many params does weight tying save?
cfg = dict(vocab_size=65, d_model=64, num_layers=3, num_heads=4, hidden_dim=256, max_seq_len=128)
tied = MyGPT(**cfg)
print("tied params:", sum(p.numel() for p in tied.parameters()))
# TODO: make an untied version and compare.